In [ ]:
# ============================================================
# FORCE PLATE FEATURE EXTRACTION — Full Feature Set
# Features: COP displacement (AP/ML), COP velocity,
#           GRF variability, sway path length
# Output: fp_features.csv saved to Google Drive
# ============================================================

import numpy as np
import pandas as pd
import os

# ── 1. Paths ──────────────────────────────────────────────────────────────
dataset_path = (
    '/content/501Project_Dataset/'
    'multimodal-synchronized-motion-capture-force-plate-and-radar-'
    'dataset-of-the-one-legged-stand-test-for-fall-risk-assessment-1.0'
)
output_path   = '/content/drive/MyDrive/fp_features.csv'

# ── 2. Load OLST attempts ─────────────────────────────────────────────────
olst_df = pd.read_csv(f'{dataset_path}/Metadata/OLST_Attempts.csv')
olst_df['participant_id'] = olst_df['OLST_attempt_id'].str.slice(0, 2).astype(int)
olst_df['movement_code']  = olst_df['RADAR_capture'].str.split('_').str[1]
print(f'✓ Loaded OLST_Attempts.csv — {len(olst_df)} attempts')

# ── 3. Feature functions ──────────────────────────────────────────────────

def cop_displacement_features(window):
    """
    COP displacement in anteroposterior (AP = COP_Y) and
    mediolateral (ML = COP_X) directions.
    Returns range, mean, and std for each axis.
    """
    if 'COP_X' not in window.columns or 'COP_Y' not in window.columns:
        return {}
    ap = window['COP_Y']
    ml = window['COP_X']
    return {
        'COP_AP_range_mm': ap.max() - ap.min(),
        'COP_AP_mean_mm':  ap.mean(),
        'COP_AP_sd_mm':    ap.std(),
        'COP_ML_range_mm': ml.max() - ml.min(),
        'COP_ML_mean_mm':  ml.mean(),
        'COP_ML_sd_mm':    ml.std(),
    }


def cop_velocity_features(window):
    """
    COP velocity features using pre-computed COP_speed column (mm/s).
    Also recomputes instantaneous velocity from raw COP_X/Y positions.
    """
    if 'COP_speed' not in window.columns:
        return {}
    speed = window['COP_speed']
    dx = window['COP_X'].diff()
    dy = window['COP_Y'].diff()
    dt = window['time'].diff()
    computed_speed = np.sqrt(dx**2 + dy**2) / dt
    return {
        'COP_speed_mean_mm_s':     speed.mean(),
        'COP_speed_sd_mm_s':       speed.std(),
        'COP_speed_max_mm_s':      speed.max(),
        'COP_speed_computed_mean': computed_speed.mean(),
    }


def grf_variability_features(window):
    """
    Ground reaction force variability from vertical force (Force_Z).
    SD is the primary metric; CV normalises by mean bodyweight.
    """
    if 'Force_Z' not in window.columns:
        return {}
    fz      = window['Force_Z']
    mean_fz = fz.mean()
    sd_fz   = fz.std()
    cv_fz   = (sd_fz / mean_fz * 100) if mean_fz != 0 else np.nan
    return {
        'GRF_Z_mean_N':  mean_fz,
        'GRF_Z_sd_N':    sd_fz,
        'GRF_Z_cv_pct':  cv_fz,
        'GRF_Z_range_N': fz.max() - fz.min(),
    }


def sway_path_features(window):
    """
    Total COP path length = cumulative sum of Euclidean distances
    between successive samples. Also returns path normalised by duration.
    """
    if 'COP_X' not in window.columns or 'COP_Y' not in window.columns:
        return {}
    dx         = window['COP_X'].diff()
    dy         = window['COP_Y'].diff()
    step_dist  = np.sqrt(dx**2 + dy**2).fillna(0)
    total_path = step_dist.sum()
    duration   = window['time'].iloc[-1] - window['time'].iloc[0]
    path_per_s = total_path / duration if duration > 0 else np.nan
    return {
        'sway_path_length_mm':       total_path,
        'sway_path_per_second_mm_s': path_per_s,
        'trial_duration_s':          duration,
    }


def extract_all_features(window, stance, lifted, attempt_id,
                          participant_id, label):
    """Combine all feature groups into one row."""
    if len(window) < 10:
        return None

    feats = {
        'OLST_attempt_id': attempt_id,
        'participant_id':  participant_id,
        'stance_leg':      stance,
        'lifted_leg':      lifted,
        'label':           label,
    }
    feats.update(cop_displacement_features(window))
    feats.update(cop_velocity_features(window))
    feats.update(grf_variability_features(window))
    feats.update(sway_path_features(window))
    return feats


# ── 4. Main extraction loop ───────────────────────────────────────────────
all_features = []
skipped      = 0

for idx, row in olst_df.iterrows():

    if (idx + 1) % 100 == 0:
        print(f'  Processing {idx+1}/{len(olst_df)}...')

    attempt_id     = row['OLST_attempt_id']
    participant_id = row['participant_id']
    radar_capture  = row['RADAR_capture']
    movement_code  = row['movement_code']
    t_foot_up      = row['t_foot_up']
    t_stable       = row['t_stable']
    t_break        = row['t_break']
    t_end          = row['t_end']

    stance = movement_code[-1]           # 'L' or 'R'
    lifted = 'R' if stance == 'L' else 'L'

    # Derive force plate filename from RADAR_capture
    # e.g. 01_MNTRL_RR_V1 → 01_MNTRL_FP_V1_{side}.csv
    fp_base = radar_capture.replace('_RR_', '_FP_')
    fp_side = 'left' if stance == 'L' else 'right'
    fp_filename = f'{fp_base}_{fp_side}.csv'
    fp_file = f'{dataset_path}/Raw/ForcePlate/{participant_id:02d}/{fp_filename}'

    if not os.path.exists(fp_file):
        skipped += 1
        continue
    try:
        fp = pd.read_csv(fp_file)
    except Exception:
        skipped += 1
        continue

    # ── STABLE: t_stable → t_break (or t_end) ────────────────────
    if pd.notna(t_stable):
        t_stop = t_break if pd.notna(t_break) else t_end
        window = fp[(fp['time'] >= t_stable) & (fp['time'] <= t_stop)]
        feat   = extract_all_features(window, stance, lifted,
                                      attempt_id, participant_id,
                                      label='STABLE')
        if feat:
            all_features.append(feat)

    # ── UNSTABLE: failed attempts only (t_stable is NaN) ──────────
    if pd.isna(t_stable):
        window = fp[(fp['time'] >= t_foot_up) & (fp['time'] <= t_end)]
        feat   = extract_all_features(window, stance, lifted,
                                      attempt_id, participant_id,
                                      label='UNSTABLE')
        if feat:
            all_features.append(feat)

# ── 5. Save output ────────────────────────────────────────────────────────
fp_features_df = pd.DataFrame(all_features)

print(f'\n✓ Extraction complete!')
print(f'  Total rows  : {len(fp_features_df)}')
print(f'  STABLE      : {(fp_features_df["label"] == "STABLE").sum()}')
print(f'  UNSTABLE    : {(fp_features_df["label"] == "UNSTABLE").sum()}')
print(f'  Skipped     : {skipped}')
print(f'\nFeature columns ({len(fp_features_df.columns)}):')
print(fp_features_df.columns.tolist())

fp_features_df.to_csv(output_path, index=False)
print(f'\n✓ Saved to: {output_path}')